In [1]:
import pandas as pd
import numpy as np
import pickle

In [2]:
from google.colab import files
uploaded = files.upload()

Saving insurance_pre.csv to insurance_pre.csv


In [3]:
import io
dataset = pd.read_csv(io.BytesIO(next(iter(uploaded.values()))))

In [4]:
dataset

,age,sex,bmi,children,smoker,charges
0,19,female,27.900,0,yes,16884.92400
1,18,male,33.770,1,no,1725.55230
2,28,male,33.000,3,no,4449.46200
3,33,male,22.705,0,no,21984.47061
4,32,male,28.880,0,no,3866.85520
...,...,...,...,...,...,...
1333,50,male,30.970,3,no,10600.54830
1334,18,female,31.920,0,no,2205.98080
1335,18,female,36.850,0,no,1629.83350
1336,21,female,25.800,0,no,2007.94500


In [5]:
dataset.shape

(1338, 6)

In [6]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(2)
memory usage: 62.8+ KB


In [7]:
dataset = pd.get_dummies(dataset, drop_first=True)

In [8]:
independent = dataset.drop('charges', axis=1)

dependent = dataset[['charges']]

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    independent, dependent, test_size=0.30, random_state=0
)

In [10]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=0),
    "Random Forest": RandomForestRegressor(n_estimators=50, random_state=0)
}

scores = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    scores[name] = r2_score(y_test, y_pred)

results = pd.DataFrame(list(scores.items()), columns=['Model', 'R2 Score'])

results

/usr/local/lib/python3.13/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


,Model,R2 Score
0,Linear Regression,0.789479
1,Decision Tree,0.695722
2,Random Forest,0.849833


In [11]:
best_model_name = max(scores, key=scores.get)

final_model = models[best_model_name]

print("Final Model:", best_model_name)
print("R2 Score:", scores[best_model_name])

pickle.dump(final_model, open("finalized_insurance_model.sav", "wb"))

Final Model: Random Forest
R2 Score: 0.8498329315421834
